# PySpark Big Data Processing — Log Pipeline + Pandas Fallback

**Auteur :** Emmanuel TSAGUE — EDF MAD EDVANCE | Datascientest 2024  
**Données :** simulées — 100 000 événements serveur

> **Note :** Ce notebook fonctionne avec ou sans PySpark.  
> Avec PySpark installé : exécution distribuée.  
> Sans PySpark : pandas fallback avec la même logique.

### Plan
1. Génération des logs serveur
2. Pipeline Spark (ou pandas fallback)
3. Agrégations par service et heure
4. Window functions (cumul, MA)
5. Latences P95/P99
6. Dashboard monitoring

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_generation import load_or_generate
from src.spark_pipeline import (run_spark_pipeline, run_pandas_pipeline,
                                 log_volume_by_level, error_rate_by_region,
                                 plot_spark_results, SPARK_AVAILABLE)

print(f'PySpark disponible : {SPARK_AVAILABLE}')

## 1. Chargement des Logs

In [ ]:
df = load_or_generate('../data_sample/server_logs_simulated.csv')
print(f'Événements : {len(df):,}')
print(f'Services   : {df.service.unique()}')
print(df.head())

## 2. Exécution du Pipeline

In [ ]:
if SPARK_AVAILABLE:
    print('Utilisation de PySpark...')
    hourly, slow = run_spark_pipeline('../data_sample/server_logs_simulated.csv')
else:
    print('Fallback pandas (même logique de transformation)...')
    hourly, slow = run_pandas_pipeline(df)
    
print(f'Hourly agg shape : {hourly.shape}')
print(hourly.head())

## 3. Taux d'Erreur par Service

In [ ]:
err_svc = (df.groupby('service')['is_error'].mean() * 100).sort_values(ascending=False)
print('Taux d\'erreur par service (%) :')
print(err_svc.round(2))

## 4. Volume par Niveau de Log

In [ ]:
vol = log_volume_by_level(df)
print(vol)

## 5. Erreurs par Région

In [ ]:
region_err = error_rate_by_region(df)
print(region_err)

## 6. Latences P95 / P99 par Service

In [ ]:
print('Latences P95/P99 :')
print(slow)

## 7. Dashboard Monitoring

In [ ]:
plot_spark_results(df, hourly, slow)

## Synthèse

| Transformation | Spark | Pandas Équivalent |
|----------------|-------|------------------|
| Filtre + cast | `.filter()` / `.withColumn()` | `.query()` / `.astype()` |
| GROUP BY | `.groupBy().agg()` | `.groupby().agg()` |
| Window function | `Window.partitionBy().orderBy()` | `.groupby().cumsum()` |
| P95/P99 | `percentile_approx()` | `.quantile()` |

*Données simulées (seed=42) — Emmanuel TSAGUE*